# Répondre aux questions sur la presse de 1914

Un assistant qui répond aux questions à partir des 64 000 passages de la presse de 1914, en citant ses sources (un RAG), et son évaluation sur 20 questions dont on connaît la réponse et le passage qui la donne.

1. La recherche : hybride, puis reclassement (les briques du TP de recherche de produits)
2. Évaluer la recherche
3. Générer une réponse sourcée, ou refuser
4. Changer un réglage, et mesurer
5. Bonus : un LLM juge la fidélité des réponses

Ce notebook a besoin d'un GPU : *Exécution > Modifier le type d'exécution > GPU T4*.

## Données et modèles

Les passages, leurs *embeddings* déjà calculés (512 dimensions), les questions ; Ollama s'installe et télécharge le modèle en arrière-plan.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/embeddings-512.npy
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-qa/questions.jsonl
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
%pip install -q bm25s

import os
import subprocess

MODEL = "qwen3:4b-instruct-2507-q4_K_M"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": "8", "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
pull = subprocess.Popen(f"sleep 5 && ollama pull {MODEL}", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

## 1. La recherche

Les fonctions de recherche sont données : ce sont celles du TP de recherche de produits. `search(question, k)` renvoie les indices des `k` meilleurs passages : BM25 et recherche dense (100 candidats chacun), fusionnés par RRF, puis les 30 premiers reclassés par le cross-encodeur (`rerank=False` pour s'en passer). Lancez la cellule (une à deux minutes : elle attend aussi le LLM, et le charge sur le GPU avant les autres modèles), et essayez une question.

In [ ]:
# bm25s se sert de JAX s'il est installé (c'est le cas sur Colab), et JAX réserve d'office
# 75 % de la mémoire du GPU : on le cantonne au CPU, le GPU est pour les modèles.
os.environ["JAX_PLATFORMS"] = "cpu"

from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import bm25s
import numpy as np
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel
from sentence_transformers import CrossEncoder, SentenceTransformer

pull.wait()  # le modèle est téléchargé
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# Le LLM d'abord : un premier appel le charge sur le GPU, où il reste, avant les autres modèles.
client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Bonjour"}], max_tokens=1)
print(MODEL, "prêt")

passages = pd.read_parquet("passages.parquet")
vectors = np.load("embeddings-512.npy").astype(np.float32)
questions = pd.read_json("questions.jsonl", lines=True)

embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device="cuda", truncate_dim=512).half()
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda", max_length=512)
reranker.model.half()
retriever = bm25s.BM25()
retriever.index(bm25s.tokenize(passages.text.tolist(), show_progress=False), show_progress=False)


def rrf(*rankings, k: int = 60) -> list[int]:
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc] += 1 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)


def search(question: str, k: int = 5, rerank: bool = True) -> list[int]:
    lexical = retriever.retrieve(bm25s.tokenize([question], show_progress=False), k=100, show_progress=False)[0][0]
    query = embedder.encode_query(question).astype(np.float32)
    query /= np.linalg.norm(query)
    dense = np.argsort(-(vectors @ query))[:100]
    candidates = rrf(lexical.tolist(), dense.tolist())
    if not rerank:
        return candidates[:k]
    top = candidates[:30]
    scores = reranker.predict([(question, passages.text[i]) for i in top], batch_size=32)
    return [top[i] for i in np.argsort(-scores)][:k]


for i in search("Où Jaurès a-t-il été assassiné ?", k=3):
    print(passages.id[i], passages.text[i][:200])

## 2. Évaluer la recherche

Pour chacune des 18 questions qui ont une réponse dans le corpus, `questions.passage_ids` donne le passage qui la contient. Écrivez `hit_rate(k, rerank)` : la part des questions dont un passage attendu est parmi les `k` premiers résultats. Comparez avec et sans reclassement, à `k=5`.

In [ ]:
# Votre code ici

### Solution

In [ ]:
answerable = questions[questions.passage_ids.str.len() > 0]


def hit_rate(k: int = 5, rerank: bool = True) -> float:
    hits = [
        bool(set(passages.id[search(row.question, k, rerank)]) & set(row.passage_ids))
        for row in answerable.itertuples()
    ]
    return float(np.mean(hits))


for rerank in (False, True):
    print(f"reclassement={rerank} : {hit_rate(5, rerank):.0%} des passages attendus dans les 5 premiers")

## 3. Générer une réponse sourcée, ou refuser

Écrivez `answer(question)` qui :

- retrouve les 5 meilleurs passages, numérotés de 1 à 5 dans le prompt, avec leur date et leur journal
- demande au modèle une réponse au format `Answer` (ci-dessous) : la réponse, les numéros des passages qui la justifient, et `found=False` si les passages ne contiennent pas la réponse
- consigne : ne répondre **qu'à partir des passages**, en français, en une ou deux phrases

Lancez-la sur les 20 questions en parallèle, et comparez ses réponses aux réponses de référence. Les deux questions sans réponse sont-elles refusées ?

In [ ]:
class Answer(BaseModel):
    found: bool
    answer: str
    citations: list[int]

In [ ]:
# Votre code ici

### Solution

In [ ]:
INSTRUCTIONS = """Tu réponds aux questions sur la presse française de 1914, uniquement à partir
des passages fournis, numérotés. Réponds en français, en une ou deux phrases, et mets dans
citations les numéros des passages qui justifient ta réponse (pas dans la réponse elle-même).
Si les passages ne contiennent pas la réponse, mets found à false et n'invente rien."""


def answer(question: str, k: int = 5) -> dict:
    found = search(question, k)
    context = "\n\n".join(
        f"[{n}] ({passages.newspaper[i]}, {passages.date[i]:%d/%m/%Y}) {passages.text[i]}"
        for n, i in enumerate(found, start=1)
    )
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": INSTRUCTIONS},
            {"role": "user", "content": f"Passages :\n{context}\n\nQuestion : {question}"},
        ],
        response_format=Answer,
        temperature=0,
    )
    result = response.choices[0].message.parsed.model_dump()
    result["sources"] = [passages.id[found[n - 1]] for n in result["citations"] if 1 <= n <= len(found)]
    return result


with ThreadPoolExecutor(8) as pool:
    answers = list(pool.map(answer, questions.question))
for row, result in zip(questions.itertuples(), answers):
    print(f"Q{row.id} {row.question}\n  référence : {row.answer}\n  réponse   : {result['answer'] if result['found'] else '(refus)'}\n")

## 4. Changer un réglage, et mesurer

Un RAG se règle comme un modèle : en mesurant. Mesurez le taux de passages trouvés pour `k` = 1, 3, 5 et 10. Quel `k` choisiriez-vous, sachant que chaque passage ajouté allonge le prompt (coût, latence) ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
for k in (1, 3, 5, 10):
    print(f"k={k:2} : {hit_rate(k):.0%}")

## 5. Bonus : un LLM juge la fidélité

Pour chaque réponse, demandez au modèle de juger, à partir des passages cités et de la réponse de référence : la réponse est-elle **fidèle** aux passages (rien d'inventé) ? Est-elle **correcte** ? Comptez. Pourquoi faut-il se méfier d'un modèle qui juge ses propres réponses ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
class Verdict(BaseModel):
    faithful: bool
    correct: bool


def judge(question: str, reference: str | None, result: dict) -> dict:
    cited = "\n\n".join(passages.set_index("id").text[source] for source in result["sources"])
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": (
            f"Question : {question}\nRéponse de référence : {reference or 'aucune, la question est sans réponse'}\n"
            f"Passages cités :\n{cited or '(aucun)'}\n\nRéponse à juger : "
            f"{result['answer'] if result['found'] else '(refus)'}\n\n"
            "faithful : la réponse ne dit rien que les passages cités ne disent. "
            "correct : la réponse dit la même chose que la référence (un refus est correct si la question est sans réponse).\n"
            # Le format rappelé : le modèle ne voit pas le schéma, et juge moins bien sans.
            'Réponds par un objet JSON : {"faithful": true ou false, "correct": true ou false}'
        )}],
        response_format=Verdict,
        temperature=0,
    )
    return response.choices[0].message.parsed.model_dump()


with ThreadPoolExecutor(8) as pool:
    verdicts = list(pool.map(judge, questions.question, questions.answer, answers))
print(pd.DataFrame(verdicts).mean())

# Le même modèle génère et juge : il tend à approuver ses propres réponses (biais
# d'auto-préférence). En pratique, on juge avec un autre modèle, plus fort, et on vérifie
# un échantillon des verdicts à la main.